
# SQPT Data Challenge — Reusable Time-Series Forecasting Notebook

This notebook is designed for a timed onsite data challenge, especially electricity / energy forecasting tasks.

**Core workflow**
1. Load and inspect data
2. Fast EDA
3. Time-series feature engineering
4. Chronological train / validation / test split
5. Naive benchmark
6. Ridge regression
7. Random Forest
8. Optional XGBoost
9. RMSE + directional hit rate
10. Feature importance / interpretation
11. Validation plots
12. Optional time-series cross-validation

> Tomorrow, start by editing only the **CONFIG** cell below.


In [ ]:

# ============================================================
# 0. IMPORTS
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1. CONFIG — change this first tomorrow

In [ ]:

# ============================================================
# 1. CONFIG
# ============================================================

# ---- REQUIRED ----
FILE_PATH = "data.csv"          # e.g. "energy_data.csv"
TARGET = "target"               # target column to forecast

# ---- TIME COLUMN ----
TIME_COL = "datetime"           # set None if no time column

# ---- OPTIONAL ----
ID_COL = None                   # e.g. "id"
DROP_COLS = []                  # e.g. ["unnecessary_col"]

# If True, assume time-series forecasting
IS_TIME_SERIES = True

# Split ratios
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Feature engineering
USE_TIME_FEATURES = True
USE_TARGET_LAGS = True
USE_TARGET_ROLLING = True
USE_CROSS_SOURCE_LAGS = True

# IMPORTANT:
# For HOURLY energy data, common lags are 1, 2, 3, 24, 168
# For DAILY data, consider 1, 2, 7, 14, 30
TARGET_LAGS = [1, 2, 3, 24, 168]
ROLLING_WINDOWS = [6, 24, 168]
CROSS_SOURCE_LAGS = [1, 24]

# Modeling
RIDGE_ALPHA = 1.0
RF_N_ESTIMATORS = 200
RF_MAX_DEPTH = 10
RANDOM_STATE = 42

# Optional XGBoost
RUN_XGBOOST = False


## 2. Load data

In [ ]:

# ============================================================
# 2. LOAD DATA
# ============================================================

if FILE_PATH.lower().endswith(".csv"):
    df = pd.read_csv(FILE_PATH)
elif FILE_PATH.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(FILE_PATH)
else:
    raise ValueError("Unsupported file format. Use CSV or Excel.")

print("Raw shape:", df.shape)
display(df.head())


## 3. Basic structure and cleaning

In [ ]:

# ============================================================
# 3. BASIC CLEANING
# ============================================================

# Drop duplicates
df = df.drop_duplicates().copy()

# Drop manually specified columns
df = df.drop(columns=DROP_COLS, errors="ignore")

# Optional ID index
if ID_COL is not None and ID_COL in df.columns:
    df = df.set_index(ID_COL)

# Parse time
if TIME_COL is not None and TIME_COL in df.columns:
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])
    df = df.sort_values(TIME_COL).reset_index(drop=True)

# Check target
if TARGET not in df.columns:
    raise ValueError(f"TARGET='{TARGET}' is not in the dataset.")

# Never impute target by default
df = df.dropna(subset=[TARGET]).reset_index(drop=True)

print("Clean shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values (%):")
display((df.isna().mean() * 100).sort_values(ascending=False).head(20))


## 4. Fast EDA — keep this focused

In [ ]:

# ============================================================
# 4A. SUMMARY STATISTICS
# ============================================================

print("Target summary:")
display(df[TARGET].describe())

print("Target skewness:", df[TARGET].skew())

print("\nNumeric summary:")
display(df.select_dtypes(include=np.number).describe().T)


In [ ]:

# ============================================================
# 4B. TARGET OVER TIME
# ============================================================

if TIME_COL is not None and TIME_COL in df.columns:
    plt.figure(figsize=(12, 4))
    plt.plot(df[TIME_COL], df[TARGET], linewidth=1)
    plt.title(f"{TARGET} over time")
    plt.xlabel("Time")
    plt.ylabel(TARGET)
    plt.show()


In [ ]:

# ============================================================
# 4C. TARGET DISTRIBUTION
# ============================================================

plt.figure(figsize=(7, 4))
plt.hist(df[TARGET].dropna(), bins=50)
plt.title(f"Distribution of {TARGET}")
plt.xlabel(TARGET)
plt.ylabel("Frequency")
plt.show()


In [ ]:

# ============================================================
# 4D. CORRELATION WITH TARGET
# ============================================================

numeric_df = df.select_dtypes(include=np.number)

if TARGET in numeric_df.columns:
    target_corr = (
        numeric_df.corr()[TARGET]
        .drop(TARGET)
        .sort_values(key=np.abs, ascending=False)
    )
    display(target_corr.head(20))



### What to look for in EDA
- Trend or regime changes
- Daily / weekly / monthly seasonality
- Spikes that may be economically meaningful rather than bad data
- Highly correlated energy sources
- Missing-value patterns
- Whether the target is persistent/autocorrelated


## 5. Time features

In [ ]:

# ============================================================
# 5. TIME FEATURES
# ============================================================

if USE_TIME_FEATURES and TIME_COL is not None and TIME_COL in df.columns:
    dt = df[TIME_COL]

    df["hour"] = dt.dt.hour
    df["day_of_week"] = dt.dt.dayofweek
    df["day"] = dt.dt.day
    df["month"] = dt.dt.month
    df["quarter"] = dt.dt.quarter
    df["year"] = dt.dt.year
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    # Cyclical encodings
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)


## 6. Seasonality checks

In [ ]:

# ============================================================
# 6. SEASONALITY CHECKS
# ============================================================

if "hour" in df.columns:
    hourly = df.groupby("hour")[TARGET].mean()

    plt.figure(figsize=(8, 4))
    plt.plot(hourly.index, hourly.values, marker="o")
    plt.title(f"Average {TARGET} by hour")
    plt.xlabel("Hour")
    plt.ylabel(f"Average {TARGET}")
    plt.show()

if "day_of_week" in df.columns:
    dow = df.groupby("day_of_week")[TARGET].mean()

    plt.figure(figsize=(8, 4))
    plt.plot(dow.index, dow.values, marker="o")
    plt.title(f"Average {TARGET} by day of week")
    plt.xlabel("Day of week (0=Mon)")
    plt.ylabel(f"Average {TARGET}")
    plt.show()

if "month" in df.columns:
    monthly = df.groupby("month")[TARGET].mean()

    plt.figure(figsize=(8, 4))
    plt.plot(monthly.index, monthly.values, marker="o")
    plt.title(f"Average {TARGET} by month")
    plt.xlabel("Month")
    plt.ylabel(f"Average {TARGET}")
    plt.show()


## 7. Lag and rolling features — leakage-safe

In [ ]:

# ============================================================
# 7A. TARGET LAGS
# ============================================================

if IS_TIME_SERIES and USE_TARGET_LAGS:
    for lag in TARGET_LAGS:
        df[f"{TARGET}_lag_{lag}"] = df[TARGET].shift(lag)


In [ ]:

# ============================================================
# 7B. TARGET ROLLING FEATURES
# IMPORTANT: shift first, then rolling
# ============================================================

if IS_TIME_SERIES and USE_TARGET_ROLLING:
    past_target = df[TARGET].shift(1)

    for window in ROLLING_WINDOWS:
        df[f"{TARGET}_mean_{window}"] = past_target.rolling(window).mean()
        df[f"{TARGET}_std_{window}"] = past_target.rolling(window).std()
        df[f"{TARGET}_min_{window}"] = past_target.rolling(window).min()
        df[f"{TARGET}_max_{window}"] = past_target.rolling(window).max()

    # Previous observed change
    df[f"{TARGET}_diff_1"] = df[TARGET].diff(1).shift(1)


## 8. Cross-source lag features

In [ ]:

# ============================================================
# 8. CROSS-SOURCE LAGS
# ============================================================

# Use original numeric source columns only.
# Exclude target and engineered features.
original_numeric_cols = [
    c for c in numeric_df.columns
    if c != TARGET
]

if IS_TIME_SERIES and USE_CROSS_SOURCE_LAGS:
    for col in original_numeric_cols:
        for lag in CROSS_SOURCE_LAGS:
            df[f"{col}_lag_{lag}"] = df[col].shift(lag)

print("Shape after feature engineering:", df.shape)



### Important modeling principle
If the prompt says **use all available past data**, avoid contemporaneous variables unless you are certain they would already be known at forecast time.

Lagging predictors is the safest default for a forecasting task.


## 9. Missing values after feature engineering

In [ ]:

# ============================================================
# 9. HANDLE NaNs / INF
# ============================================================

df = df.replace([np.inf, -np.inf], np.nan)

# Lag / rolling features naturally create missing values at the beginning.
# For a timed time-series challenge, dropping those rows is usually simplest.
df_model = df.dropna().reset_index(drop=True).copy()

print("Modeling shape:", df_model.shape)
print("Remaining missing values:", df_model.isna().sum().sum())


## 10. Define features and chronological split

In [ ]:

# ============================================================
# 10. FEATURE MATRIX + SPLIT
# ============================================================

exclude = [TARGET]
if TIME_COL is not None:
    exclude.append(TIME_COL)

features = [
    c for c in df_model.columns
    if c not in exclude
]

# Keep numeric features only for speed / reliability in the onsite.
X = df_model[features].select_dtypes(include=np.number).copy()
y = df_model[TARGET].copy()

features = X.columns.tolist()

n = len(df_model)

train_end = int(n * TRAIN_RATIO)
val_end = int(n * (TRAIN_RATIO + VAL_RATIO))

X_train = X.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()

X_val = X.iloc[train_end:val_end].copy()
y_val = y.iloc[train_end:val_end].copy()

X_test = X.iloc[val_end:].copy()
y_test = y.iloc[val_end:].copy()

train_df = df_model.iloc[:train_end].copy()
val_df = df_model.iloc[train_end:val_end].copy()
test_df = df_model.iloc[val_end:].copy()

print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)


## 11. Naive time-series baselines

In [ ]:

# ============================================================
# 11. NAIVE BASELINES
# ============================================================

def regression_metrics(y_true, pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "MAE": mean_absolute_error(y_true, pred),
        "R2": r2_score(y_true, pred),
    }

baseline_rows = []

# Persistence benchmark: previous observation
lag1_col = f"{TARGET}_lag_1"

if lag1_col in val_df.columns:
    pred_naive_1 = val_df[lag1_col].values
    metrics = regression_metrics(y_val, pred_naive_1)
    metrics["Model"] = "Naive lag-1"
    baseline_rows.append(metrics)

# Seasonal benchmark: previous day for hourly data
lag24_col = f"{TARGET}_lag_24"

if lag24_col in val_df.columns:
    pred_naive_24 = val_df[lag24_col].values
    metrics = regression_metrics(y_val, pred_naive_24)
    metrics["Model"] = "Naive lag-24"
    baseline_rows.append(metrics)

baseline_results = pd.DataFrame(baseline_rows)
display(baseline_results)


## 12. Ridge regression — main interpretable model

In [ ]:

# ============================================================
# 12. RIDGE
# ============================================================

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

ridge = Ridge(alpha=RIDGE_ALPHA)

ridge.fit(X_train_s, y_train)

ridge_val_pred = ridge.predict(X_val_s)

ridge_metrics = regression_metrics(y_val, ridge_val_pred)

print("Ridge validation metrics:")
print(ridge_metrics)


## 13. Random Forest — nonlinear benchmark

In [ ]:

# ============================================================
# 13. RANDOM FOREST
# ============================================================

rf = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_val_pred = rf.predict(X_val)

rf_metrics = regression_metrics(y_val, rf_val_pred)

print("Random Forest validation metrics:")
print(rf_metrics)


## 14. Optional XGBoost

In [ ]:

# ============================================================
# 14. OPTIONAL XGBOOST
# ============================================================

xgb_model = None
xgb_val_pred = None
xgb_metrics = None

if RUN_XGBOOST:
    try:
        from xgboost import XGBRegressor

        xgb_model = XGBRegressor(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        xgb_model.fit(X_train, y_train)
        xgb_val_pred = xgb_model.predict(X_val)
        xgb_metrics = regression_metrics(y_val, xgb_val_pred)

        print("XGBoost validation metrics:")
        print(xgb_metrics)

    except Exception as e:
        print("XGBoost unavailable:", e)


## 15. Compare models

In [ ]:

# ============================================================
# 15. MODEL COMPARISON
# ============================================================

rows = []

for _, row in baseline_results.iterrows():
    rows.append(row.to_dict())

rows.append({"Model": "Ridge", **ridge_metrics})
rows.append({"Model": "Random Forest", **rf_metrics})

if xgb_metrics is not None:
    rows.append({"Model": "XGBoost", **xgb_metrics})

model_results = (
    pd.DataFrame(rows)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(model_results)


## 16. Directional hit rate

In [ ]:

# ============================================================
# 16. DIRECTIONAL HIT RATE
# ============================================================

def directional_hit_rate(y_true, pred, previous_level):
    actual_change = np.asarray(y_true) - np.asarray(previous_level)
    pred_change = np.asarray(pred) - np.asarray(previous_level)

    valid = (
        np.isfinite(actual_change)
        & np.isfinite(pred_change)
    )

    return np.mean(
        np.sign(actual_change[valid])
        == np.sign(pred_change[valid])
    )

if lag1_col in val_df.columns:
    previous_level = val_df[lag1_col].values

    ridge_hit = directional_hit_rate(
        y_val.values,
        ridge_val_pred,
        previous_level
    )

    rf_hit = directional_hit_rate(
        y_val.values,
        rf_val_pred,
        previous_level
    )

    print("Ridge directional hit rate:", round(ridge_hit, 4))
    print("RF directional hit rate:   ", round(rf_hit, 4))

    if xgb_val_pred is not None:
        xgb_hit = directional_hit_rate(
            y_val.values,
            xgb_val_pred,
            previous_level
        )
        print("XGB directional hit rate:  ", round(xgb_hit, 4))


## 17. Feature importance / interpretation

In [ ]:

# ============================================================
# 17A. RIDGE COEFFICIENTS
# ============================================================

ridge_coef = pd.DataFrame({
    "feature": features,
    "coefficient": ridge.coef_
})

ridge_coef["abs_coefficient"] = ridge_coef["coefficient"].abs()

ridge_coef = ridge_coef.sort_values(
    "abs_coefficient",
    ascending=False
)

display(ridge_coef.head(20))


In [ ]:

# ============================================================
# 17B. RANDOM FOREST FEATURE IMPORTANCE
# ============================================================

rf_importance = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(rf_importance.head(20))


## 18. Temporal decay check

In [ ]:

# ============================================================
# 18. TEMPORAL DECAY
# ============================================================

target_lag_cols = [
    c for c in df_model.columns
    if c.startswith(f"{TARGET}_lag_")
]

if target_lag_cols:
    temporal_decay = (
        df_model[target_lag_cols + [TARGET]]
        .corr()[TARGET]
        .drop(TARGET)
        .sort_index()
    )

    display(temporal_decay)


## 19. Actual vs predicted

In [ ]:

# ============================================================
# 19A. VALIDATION FORECAST
# ============================================================

if TIME_COL is not None and TIME_COL in val_df.columns:
    x_axis = val_df[TIME_COL]
else:
    x_axis = np.arange(len(y_val))

plt.figure(figsize=(13, 5))
plt.plot(x_axis, y_val.values, label="Actual", linewidth=1.5)
plt.plot(x_axis, ridge_val_pred, label="Ridge", linewidth=1)
plt.plot(x_axis, rf_val_pred, label="Random Forest", linewidth=1)

if xgb_val_pred is not None:
    plt.plot(x_axis, xgb_val_pred, label="XGBoost", linewidth=1)

plt.title("Validation: Actual vs Predicted")
plt.xlabel("Time")
plt.ylabel(TARGET)
plt.legend()
plt.show()


In [ ]:

# ============================================================
# 19B. SCATTER: ACTUAL VS PREDICTED
# ============================================================

plt.figure(figsize=(6, 6))
plt.scatter(y_val, ridge_val_pred, alpha=0.5)

lims = [
    min(y_val.min(), ridge_val_pred.min()),
    max(y_val.max(), ridge_val_pred.max())
]

plt.plot(lims, lims, "--")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Ridge: Actual vs Predicted")
plt.show()


## 20. Residual analysis

In [ ]:

# ============================================================
# 20. RESIDUALS
# ============================================================

ridge_residuals = y_val.values - ridge_val_pred

print("Residual mean:", ridge_residuals.mean())
print("Residual std: ", ridge_residuals.std())

plt.figure(figsize=(7, 4))
plt.hist(ridge_residuals, bins=50)
plt.title("Ridge residual distribution")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.show()

plt.figure(figsize=(7, 4))
plt.scatter(ridge_val_pred, ridge_residuals, alpha=0.5)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residual")
plt.title("Residuals vs Predicted")
plt.show()


## 21. Error inspection — useful for presentation

In [ ]:

# ============================================================
# 21. LARGEST ERRORS
# ============================================================

error_df = val_df[[c for c in [TIME_COL, TARGET] if c is not None and c in val_df.columns]].copy()

error_df["prediction"] = ridge_val_pred
error_df["residual"] = y_val.values - ridge_val_pred
error_df["abs_error"] = np.abs(error_df["residual"])

display(
    error_df
    .sort_values("abs_error", ascending=False)
    .head(20)
)


## 22. Optional expanding-window cross-validation

In [ ]:

# ============================================================
# 22. OPTIONAL TIME-SERIES CV
# ============================================================

RUN_TIME_SERIES_CV = False

if RUN_TIME_SERIES_CV:
    tscv = TimeSeriesSplit(n_splits=5)

    fold_results = []

    for fold, (tr_idx, vl_idx) in enumerate(tscv.split(X), start=1):

        X_tr = X.iloc[tr_idx]
        y_tr = y.iloc[tr_idx]

        X_vl = X.iloc[vl_idx]
        y_vl = y.iloc[vl_idx]

        fold_scaler = StandardScaler()
        X_tr_s = fold_scaler.fit_transform(X_tr)
        X_vl_s = fold_scaler.transform(X_vl)

        fold_model = Ridge(alpha=RIDGE_ALPHA)
        fold_model.fit(X_tr_s, y_tr)

        pred = fold_model.predict(X_vl_s)

        fold_results.append({
            "fold": fold,
            "RMSE": np.sqrt(mean_squared_error(y_vl, pred)),
            "MAE": mean_absolute_error(y_vl, pred),
            "R2": r2_score(y_vl, pred)
        })

    cv_results = pd.DataFrame(fold_results)
    display(cv_results)

    print("Average CV RMSE:", cv_results["RMSE"].mean())
    print("CV RMSE std:    ", cv_results["RMSE"].std())


## 23. Refit best model on train + validation, then evaluate once on test

In [ ]:

# ============================================================
# 23. FINAL TEST
# ============================================================

# Choose after looking at validation results.
# Default: Ridge for interpretability.
FINAL_MODEL = "Ridge"   # "Ridge", "Random Forest", "XGBoost"

X_trainval = pd.concat([X_train, X_val], axis=0)
y_trainval = pd.concat([y_train, y_val], axis=0)

if FINAL_MODEL == "Ridge":

    final_scaler = StandardScaler()

    X_trainval_s = final_scaler.fit_transform(X_trainval)
    X_test_s_final = final_scaler.transform(X_test)

    final_model = Ridge(alpha=RIDGE_ALPHA)
    final_model.fit(X_trainval_s, y_trainval)

    test_pred = final_model.predict(X_test_s_final)

elif FINAL_MODEL == "Random Forest":

    final_model = RandomForestRegressor(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    final_model.fit(X_trainval, y_trainval)
    test_pred = final_model.predict(X_test)

elif FINAL_MODEL == "XGBoost":

    if xgb_model is None:
        raise ValueError("Set RUN_XGBOOST=True and rerun the XGBoost section first.")

    from xgboost import XGBRegressor

    final_model = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    final_model.fit(X_trainval, y_trainval)
    test_pred = final_model.predict(X_test)

else:
    raise ValueError("Unknown FINAL_MODEL")


test_metrics = regression_metrics(y_test, test_pred)

print("FINAL TEST METRICS")
print(test_metrics)

if lag1_col in test_df.columns:
    test_hit = directional_hit_rate(
        y_test.values,
        test_pred,
        test_df[lag1_col].values
    )

    print("Directional hit rate:", test_hit)


## 24. Final test plot

In [ ]:

# ============================================================
# 24. TEST FORECAST PLOT
# ============================================================

if TIME_COL is not None and TIME_COL in test_df.columns:
    x_axis = test_df[TIME_COL]
else:
    x_axis = np.arange(len(y_test))

plt.figure(figsize=(13, 5))
plt.plot(x_axis, y_test.values, label="Actual", linewidth=1.5)
plt.plot(x_axis, test_pred, label=f"Predicted ({FINAL_MODEL})", linewidth=1)
plt.title("Final Test Forecast")
plt.xlabel("Time")
plt.ylabel(TARGET)
plt.legend()
plt.show()


## 25. Presentation-ready summary

In [ ]:

# ============================================================
# 25. SUMMARY
# ============================================================

print("DATA")
print("Rows used:", len(df_model))
print("Features:", len(features))
print()

print("SPLIT")
print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))
print()

print("VALIDATION MODEL COMPARISON")
display(model_results)
print()

print("FINAL MODEL:", FINAL_MODEL)
print("FINAL TEST METRICS:", test_metrics)

if lag1_col in test_df.columns:
    print("FINAL DIRECTIONAL HIT RATE:", test_hit)

print()
print("TOP RIDGE FEATURES")
display(ridge_coef.head(10))

print("TOP RANDOM FOREST FEATURES")
display(rf_importance.head(10))



# Presentation checklist

A clean 1-hour presentation can follow this structure:

### 1. Problem framing
- What exactly is the target?
- What is the forecast horizon?
- What information is available at prediction time?

### 2. Data quality
- Frequency and time span
- Missingness
- Unusual spikes / anomalies
- Avoid removing spikes automatically if they may be economically meaningful

### 3. EDA findings
- Trend / regimes
- Seasonality
- Autocorrelation / persistence
- Cross-source relationships

### 4. Feature engineering
- Calendar effects
- Target lags
- Rolling mean / volatility
- Cross-source lagged features
- Explicitly explain leakage prevention

### 5. Validation design
- Chronological split
- No random shuffle
- Naive persistence benchmark

### 6. Models
- Ridge for interpretability and correlated lag features
- Random Forest / XGBoost for nonlinear relationships

### 7. Evaluation
- RMSE
- Directional hit rate
- Compare against naive baseline

### 8. Interpretation
- Which signals matter most?
- Does predictive power decay with lag?
- Which energy sources contribute?

### 9. Error analysis
- When does the model fail?
- Spikes?
- Regime shifts?
- Missing exogenous variables?

### 10. Next steps
- Expanding-window CV
- Better forecast-horizon-specific features
- Weather / demand / price / outage data if available
- Regime-aware models
- More careful hyperparameter tuning

---

## One-sentence research story

> I first characterized the temporal structure of the target, then tested whether its own history and other energy sources contained incremental predictive information, and finally compared interpretable and nonlinear forecasting models against a naive time-series benchmark using leakage-safe chronological validation.



# Appendix A — Standard EDA & Model Diagnostic Charts

These cells reproduce the standard charts commonly expected in a timed data challenge.

They are intentionally kept simple and reusable.


## A1. Feature Correlation Heatmap

In [ ]:

# ============================================================
# A1. FEATURE CORRELATION HEATMAP
# ============================================================

# Use numeric columns only.
corr_df = df_model.select_dtypes(include=np.number)

# Keep target + top correlated features so the heatmap remains readable.
target_corr_abs = (
    corr_df.corr()[TARGET]
    .abs()
    .sort_values(ascending=False)
)

top_cols = target_corr_abs.head(20).index.tolist()
corr_matrix = corr_df[top_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(corr_matrix.values, aspect="auto")

ax.set_xticks(np.arange(len(top_cols)))
ax.set_yticks(np.arange(len(top_cols)))

ax.set_xticklabels(top_cols, rotation=90)
ax.set_yticklabels(top_cols)

for i in range(len(top_cols)):
    for j in range(len(top_cols)):
        ax.text(
            j, i,
            f"{corr_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center",
            fontsize=7
        )

ax.set_title("Feature Correlation Heatmap")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()



**How to interpret it**

- Look at the first row / column to see which variables are most correlated with the target.
- Look for groups of features that are highly correlated with each other.
- Strong correlation among rolling / lag variables suggests multicollinearity, which is one reason Ridge regression is useful.
- Correlation is exploratory; it does **not** by itself imply predictive causality.


## A2. Target Distribution

In [ ]:

# ============================================================
# A2. TARGET DISTRIBUTION
# ============================================================

plt.figure(figsize=(8, 5))
plt.hist(df_model[TARGET], bins=50, edgecolor="black", alpha=0.7)
plt.title(f"Distribution of {TARGET}")
plt.xlabel(TARGET)
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## A3. Log-transformed Target Distribution

In [ ]:

# ============================================================
# A3. LOG-TRANSFORMED TARGET DISTRIBUTION
# ============================================================

# log1p requires values > -1.
# Only use this plot when the target supports the transformation.

if (df_model[TARGET] > -1).all():

    log_target = np.log1p(df_model[TARGET])

    plt.figure(figsize=(8, 5))
    plt.hist(log_target, bins=50, edgecolor="black", alpha=0.7)
    plt.title(f"Log-transformed {TARGET}")
    plt.xlabel(f"log(1 + {TARGET})")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

else:
    print("Skipped: target contains values <= -1, so log1p is not directly applicable.")



Compare the original and log-transformed distributions.

If the original target is strongly right-skewed and the log version is much more symmetric, you can consider predicting the log target. Do **not** transform automatically; first check whether the transformation makes economic sense.


## A4. Target Over Time — Last 500 Observations

In [ ]:

# ============================================================
# A4. TARGET OVER TIME — LAST 500 OBSERVATIONS
# ============================================================

plot_df = df_model.tail(500)

plt.figure(figsize=(14, 5))

if TIME_COL is not None and TIME_COL in plot_df.columns:
    plt.plot(plot_df[TIME_COL], plot_df[TARGET], linewidth=1)
    plt.xlabel("Time")
else:
    plt.plot(np.arange(len(plot_df)), plot_df[TARGET], linewidth=1)
    plt.xlabel("Observation")

plt.ylabel(TARGET)
plt.title(f"{TARGET} Over Time (Last 500 Observations)")
plt.tight_layout()
plt.show()



Use this chart to identify short-run persistence, repeated patterns, spikes, volatility clusters, and regime changes. For energy data, large spikes should be investigated before being treated as errors.


## A5. Top 20 Feature Importance

In [ ]:

# ============================================================
# A5. TOP 20 RANDOM FOREST FEATURE IMPORTANCE
# ============================================================

top_fi = (
    rf_importance
    .head(20)
    .sort_values("importance", ascending=True)
)

plt.figure(figsize=(10, 8))
plt.barh(top_fi["feature"], top_fi["importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 Feature Importance")
plt.tight_layout()
plt.show()



This answers **which variables the nonlinear model relies on most**.

For a strong presentation, group the important features conceptually:
- target history
- rolling statistics
- other energy sources
- demand / weather
- calendar effects

Feature importance gives magnitude, but not necessarily the direction of the effect.


## A6. Predicted vs Actual — Test Set

In [ ]:

# ============================================================
# A6. PREDICTED VS ACTUAL — TEST SET
# ============================================================

plt.figure(figsize=(8, 7))
plt.scatter(y_test, test_pred, alpha=0.4, s=12)

low = min(y_test.min(), test_pred.min())
high = max(y_test.max(), test_pred.max())

plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Predicted vs Actual (Test Set)")
plt.tight_layout()
plt.show()



The diagonal represents perfect predictions.

- Points close to the diagonal = accurate forecasts.
- High actual values below the diagonal = the model underpredicts extreme highs.
- Low actual values above the diagonal = the model overpredicts lows.
- A compressed prediction range often means the model is shrinking forecasts toward the mean.


## A7. Residual Plot

In [ ]:

# ============================================================
# A7. RESIDUAL PLOT
# ============================================================

test_residuals = y_test.values - test_pred

plt.figure(figsize=(8, 5))
plt.scatter(test_pred, test_residuals, alpha=0.4, s=12)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.tight_layout()
plt.show()



Ideal residuals should look like random noise around zero.

Patterns may indicate:
- nonlinearity
- heteroskedasticity
- regime effects
- missing explanatory variables
- systematic under/over-prediction


## A8. Residual Distribution

In [ ]:

# ============================================================
# A8. RESIDUAL DISTRIBUTION
# ============================================================

plt.figure(figsize=(8, 5))
plt.hist(test_residuals, bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.tight_layout()
plt.show()

print("Residual mean:", test_residuals.mean())
print("Residual std: ", test_residuals.std())
print("Residual skew:", pd.Series(test_residuals).skew())



A distribution centered near zero suggests limited average bias. Heavy tails indicate occasional large forecasting errors, which are particularly important in electricity and other energy markets.


## A9. Rolling Mean and Rolling Volatility

In [ ]:

# ============================================================
# A9. ROLLING LEVEL
# ============================================================

window = 24 if len(df_model) >= 24 else max(2, len(df_model) // 10)

rolling_mean_plot = df_model[TARGET].rolling(window).mean()

plt.figure(figsize=(14, 5))

if TIME_COL is not None and TIME_COL in df_model.columns:
    plt.plot(df_model[TIME_COL], df_model[TARGET], alpha=0.35, label="Target")
    plt.plot(df_model[TIME_COL], rolling_mean_plot, linewidth=2, label=f"Rolling mean ({window})")
else:
    plt.plot(df_model[TARGET].values, alpha=0.35, label="Target")
    plt.plot(rolling_mean_plot.values, linewidth=2, label=f"Rolling mean ({window})")

plt.title(f"{TARGET}: Rolling Mean")
plt.xlabel("Time")
plt.ylabel(TARGET)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

# ============================================================
# A10. ROLLING VOLATILITY
# ============================================================

rolling_std_plot = df_model[TARGET].rolling(window).std()

plt.figure(figsize=(14, 5))

if TIME_COL is not None and TIME_COL in df_model.columns:
    plt.plot(df_model[TIME_COL], rolling_std_plot, linewidth=1)
else:
    plt.plot(rolling_std_plot.values, linewidth=1)

plt.title(f"{TARGET}: Rolling Volatility ({window})")
plt.xlabel("Time")
plt.ylabel("Rolling standard deviation")
plt.tight_layout()
plt.show()



These two plots help separate:
- **level / trend changes** from
- **volatility regime changes**.

If volatility changes materially over time, this can motivate rolling-volatility features or regime-aware modeling.
